# [SQL 실습 #09] MySQL 집계 함수 기초
## COUNT, SUM, AVG, MAX, MIN

> **학생용 실습 노트북 - TODO + 자동점검형**

이번 실습은 **SQL-9.pdf**의 흐름에 맞추어 구성했습니다.

지난 시간에는 `ORDER BY`, `LIMIT`, `OFFSET`을 이용하여 데이터를 정렬하고 필요한 개수만 조회했습니다.  
이번 시간에는 여러 행의 데이터를 **하나의 값으로 요약하는 집계 함수**를 배웁니다.

## 오늘의 학습 목표
1. 집계 함수의 의미를 이해한다.
2. `COUNT(*)`로 전체 행 개수를 구한다.
3. `COUNT(컬럼명)`과 `COUNT(*)`의 차이를 설명한다.
4. `SUM()`으로 합계를 구한다.
5. `AVG()`로 평균을 구한다.
6. `ROUND()`로 평균을 보기 좋게 출력한다.
7. `MAX()`, `MIN()`으로 최댓값과 최솟값을 구한다.
8. 여러 집계 함수를 한 번에 사용한다.
9. `WHERE`와 집계 함수를 함께 사용한다.
10. `NULL`이 집계 함수에 미치는 영향을 이해한다.
11. `IFNULL()`로 NULL을 0으로 바꾸어 계산한다.

> **핵심:** 집계 함수는 여러 개의 데이터를 하나의 값으로 요약해 주는 함수입니다.

# 1. MySQL 실습 환경 준비

Google Colab에서 실행할 수 있도록 MySQL 서버를 설치하고 시작합니다.

> 아래 셀은 수정하지 말고 실행하세요.

In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start

print("✅ MySQL 서버 준비 완료")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting MySQL database server mysqld        su: warning: cannot change directory to /nonexistent: No such file or directory
[ OK ]
✅ MySQL 서버 준비 완료


# 2. 실습 데이터 준비

SQL-9.pdf의 실습 흐름에 맞추어 `students` 테이블에 `score` 컬럼을 포함한 데이터를 준비합니다.

| id | name | grade | class_name | score |
|---:|---|---:|---|---:|
| 1 | 김부장 | 3 | 컴퓨터과 | 85 |
| 2 | 혼이 | 1 | 영혼반 | 90 |
| 3 | 라즈베리 | 2 | 서버반 | 78 |
| 4 | 파이썬 | 1 | 프로그래밍반 | 88 |
| 5 | 김코딩 | 1 | 컴퓨터과 | 95 |
| 6 | 데이터왕 | 1 | 컴퓨터과 | 82 |
| 7 | 이몽룡 | 2 | 데이터베이스반 | 91 |
| 8 | 홍길동 | 2 | 컴퓨터과 | 76 |
| 9 | 이순신 | 3 | 데이터베이스반 | 89 |
| 10 | 정렬이 | 1 | SQL반 | NULL |

`정렬이` 학생의 점수는 일부러 `NULL`로 두었습니다.  
뒤에서 `COUNT(*)`와 `COUNT(score)`의 차이를 확인합니다.

In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;

DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    score INT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name, score) VALUES
('김부장', 3, '컴퓨터과', 85),
('혼이', 1, '영혼반', 90),
('라즈베리', 2, '서버반', 78),
('파이썬', 1, '프로그래밍반', 88),
('김코딩', 1, '컴퓨터과', 95),
('데이터왕', 1, '컴퓨터과', 82),
('이몽룡', 2, '데이터베이스반', 91),
('홍길동', 2, '컴퓨터과', 76),
('이순신', 3, '데이터베이스반', 89),
('정렬이', 1, 'SQL반', NULL);

SELECT id, name, grade, class_name, score
FROM students
ORDER BY id;
SQL

id	name	grade	class_name	score
1	김부장	3	컴퓨터과	85
2	혼이	1	영혼반	90
3	라즈베리	2	서버반	78
4	파이썬	1	프로그래밍반	88
5	김코딩	1	컴퓨터과	95
6	데이터왕	1	컴퓨터과	82
7	이몽룡	2	데이터베이스반	91
8	홍길동	2	컴퓨터과	76
9	이순신	3	데이터베이스반	89
10	정렬이	1	SQL반	NULL


# 3. SQL 실행 및 자동점검 도우미

- `TODO`가 남아 있으면 실행하지 않습니다.
- 학생 실습 셀은 조회 전용입니다.
- 자동점검은 SQL 실행 결과를 정답 결과와 비교합니다.

> 아래 셀은 수정하지 마세요.

In [ ]:
import subprocess
import textwrap
import re

TODO_STATUS = {}

def _mysql(sql, no_header=False):
    cmd = ["sudo", "mysql", "--batch", "--raw"]
    if no_header:
        cmd.append("-N")
    p = subprocess.run(
        cmd,
        input=textwrap.dedent(sql).strip(),
        text=True,
        capture_output=True
    )
    return p.returncode == 0, p.stdout.strip(), p.stderr.strip()

def _real(sql):
    return "\\n".join(
        line for line in textwrap.dedent(sql).splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ).strip()

def run_sql(sql):
    real = _real(sql)
    if not real:
        print("⚠️ SQL이 비어 있습니다.")
        return False
    if "TODO" in real:
        print("⚠️ TODO가 남아 있습니다. 먼저 완성하세요.")
        return False
    if re.search(r"\\b(INSERT|UPDATE|DELETE|DROP|TRUNCATE|ALTER|CREATE)\\b", real, re.I):
        print("❌ 이 TODO 셀은 조회 전용입니다.")
        return False

    ok, out, err = _mysql(sql)
    if out:
        print(out)
    if not ok:
        print("❌ MySQL 오류")
        print(err)
        return False

    print("✅ 조회 완료")
    return True

def _result(sql):
    ok, out, err = _mysql(sql, no_header=True)
    if not ok:
        raise RuntimeError(err)
    return out.splitlines() if out else []

def check_todo(todo_id, student_sql, expected_sql):
    real = _real(student_sql)
    if not real or "TODO" in real:
        TODO_STATUS[todo_id] = False
        print(f"🔎 TODO {todo_id} 자동점검")
        print("❌ TODO를 먼저 완성하세요.")
        return False

    try:
        actual = _result(student_sql)
        expected = _result(expected_sql)
    except Exception as e:
        TODO_STATUS[todo_id] = False
        print(f"🔎 TODO {todo_id} 자동점검")
        print("❌ SQL 실행 오류:", e)
        return False

    passed = (actual == expected)
    TODO_STATUS[todo_id] = passed
    print(f"🔎 TODO {todo_id} 자동점검")
    print("✅ 통과했습니다." if passed else "❌ 결과를 다시 확인하세요.")
    return passed

def show_progress(total):
    print("===== TODO 진행 현황 =====")
    passed = 0
    for i in range(1, total + 1):
        state = TODO_STATUS.get(i)
        mark = "✅" if state is True else ("❌" if state is False else "⬜")
        if state is True:
            passed += 1
        print(f"{mark} TODO {i}")
    print(f"\\n진행률: {passed}/{total}")

# 4. COUNT로 전체 학생 수 구하기

## TODO 1

전체 학생 수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT count(*) AS 전체학생수
FROM students;
'''
run_sql(sql)

전체학생수
10
✅ 조회 완료


True

In [ ]:
check_todo(
    1,
    sql,
    'USE school; SELECT COUNT(*) AS 전체학생수 FROM students;'
)

🔎 TODO 1 자동점검
✅ 통과했습니다.


True

# 5. COUNT(score)로 점수 입력 학생 수 구하기

## TODO 2

`score`가 입력된 학생 수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT count(score) AS 점수입력학생수
FROM students;
'''
run_sql(sql)

점수입력학생수
9
✅ 조회 완료


True

In [ ]:
check_todo(
    2,
    sql,
    'USE school; SELECT COUNT(score) AS 점수입력학생수 FROM students;'
)

🔎 TODO 2 자동점검
✅ 통과했습니다.


True

# 6. SUM으로 점수 합계 구하기

## TODO 3

전체 학생의 점수 합계를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT sum(score) AS 점수합계
FROM students;
'''
run_sql(sql)

점수합계
774
✅ 조회 완료


True

In [ ]:
check_todo(
    3,
    sql,
    'USE school; SELECT SUM(score) AS 점수합계 FROM students;'
)

🔎 TODO 3 자동점검
✅ 통과했습니다.


True

# 7. AVG로 평균 구하기

## TODO 4

전체 학생의 평균 점수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT avg(score) AS 평균점수
FROM students;
'''
run_sql(sql)

평균점수
86.0000
✅ 조회 완료


True

In [ ]:
check_todo(
    4,
    sql,
    'USE school; SELECT AVG(score) AS 평균점수 FROM students;'
)

🔎 TODO 4 자동점검
✅ 통과했습니다.


True

# 8. ROUND로 평균 보기 좋게 출력하기

## TODO 5

평균 점수를 소수점 첫째 자리까지 출력하세요.

In [ ]:
sql = '''
USE school;

SELECT round(avg(score),1) AS 평균점수
FROM students;
'''
run_sql(sql)

평균점수
86.0
✅ 조회 완료


True

In [ ]:
check_todo(
    5,
    sql,
    'USE school; SELECT ROUND(AVG(score), 1) AS 평균점수 FROM students;'
)

🔎 TODO 5 자동점검
✅ 통과했습니다.


True

# 9. MAX로 최고 점수 구하기

## TODO 6

가장 높은 점수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 최고점수
FROM students;
'''
run_sql(sql)

In [ ]:
check_todo(
    6,
    sql,
    'USE school; SELECT MAX(score) AS 최고점수 FROM students;'
)

# 10. MIN으로 최저 점수 구하기

## TODO 7

가장 낮은 점수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 최저점수
FROM students;
'''
run_sql(sql)

In [ ]:
check_todo(
    7,
    sql,
    'USE school; SELECT MIN(score) AS 최저점수 FROM students;'
)

# 11. COUNT(*)와 COUNT(score)의 차이

`COUNT(*)`
- 전체 행의 개수를 셉니다.
- NULL 여부와 관계없이 행 자체를 셉니다.

`COUNT(score)`
- `score` 값이 NULL이 아닌 행만 셉니다.

현재 학생은 10명이지만 한 학생의 점수가 NULL이므로 결과는 다음과 같습니다.

- `COUNT(*)` → 10
- `COUNT(score)` → 9

## TODO 8

두 값을 한 번에 조회하세요.

In [ ]:
sql = '''
USE school;

SELECT
    TODO AS 전체학생수,
    TODO AS 점수입력학생수
FROM students;
'''
run_sql(sql)

In [ ]:
check_todo(
    8,
    sql,
    'USE school; SELECT COUNT(*) AS 전체학생수, COUNT(score) AS 점수입력학생수 FROM students;'
)

# 12. 여러 집계 함수 한 번에 사용하기

한 번의 SELECT문에서 여러 집계 함수를 함께 사용할 수 있습니다.

## TODO 9

다음을 한 번에 출력하세요.

- 전체학생수
- 점수입력수
- 점수합계
- 평균점수(소수점 첫째 자리)
- 최고점수
- 최저점수

In [ ]:
sql = '''
USE school;

SELECT
    TODO AS 전체학생수,
    TODO AS 점수입력수,
    TODO AS 점수합계,
    TODO AS 평균점수,
    TODO AS 최고점수,
    TODO AS 최저점수
FROM students;
'''
run_sql(sql)

In [ ]:
check_todo(
    9,
    sql,
    'USE school; SELECT COUNT(*) AS 전체학생수, COUNT(score) AS 점수입력수, SUM(score) AS 점수합계, ROUND(AVG(score),1) AS 평균점수, MAX(score) AS 최고점수, MIN(score) AS 최저점수 FROM students;'
)

# 1학년 학생 수 구하기

## TODO 10

1학년 학생 수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 1학년학생수
FROM students
WHERE TODO;
'''
run_sql(sql)

In [ ]:
check_todo(
    10,
    sql,
    'USE school; SELECT COUNT(*) AS 1학년학생수 FROM students WHERE grade = 1;'
)

# 1학년 평균 점수 구하기

## TODO 11

1학년 학생의 평균 점수를 소수점 첫째 자리까지 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 1학년평균
FROM students
WHERE TODO;
'''
run_sql(sql)

In [ ]:
check_todo(
    11,
    sql,
    'USE school; SELECT ROUND(AVG(score),1) AS 1학년평균 FROM students WHERE grade = 1;'
)

# 2학년 최고 점수 구하기

## TODO 12

2학년 학생의 최고 점수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 2학년최고점수
FROM students
WHERE TODO;
'''
run_sql(sql)

In [ ]:
check_todo(
    12,
    sql,
    'USE school; SELECT MAX(score) AS 2학년최고점수 FROM students WHERE grade = 2;'
)

# 3학년 최저 점수 구하기

## TODO 13

3학년 학생의 최저 점수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 3학년최저점수
FROM students
WHERE TODO;
'''
run_sql(sql)

In [ ]:
check_todo(
    13,
    sql,
    'USE school; SELECT MIN(score) AS 3학년최저점수 FROM students WHERE grade = 3;'
)

# NULL을 0으로 보고 평균 구하기

## TODO 14

`IFNULL(score, 0)`을 사용하여 NULL 점수를 0점으로 보고 전체 평균을 소수점 첫째 자리까지 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS NULL포함평균
FROM students;
'''
run_sql(sql)

In [ ]:
check_todo(
    14,
    sql,
    'USE school; SELECT ROUND(AVG(IFNULL(score,0)),1) AS NULL포함평균 FROM students;'
)

# 80점 이상 학생 수 구하기

## TODO 15

80점 이상인 학생 수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 80점이상학생수
FROM students
WHERE TODO;
'''
run_sql(sql)

In [ ]:
check_todo(
    15,
    sql,
    'USE school; SELECT COUNT(*) AS 80점이상학생수 FROM students WHERE score >= 80;'
)

# 90점 이상 평균 점수 구하기

## TODO 16

90점 이상인 학생들의 평균 점수를 소수점 첫째 자리까지 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 90점이상평균
FROM students
WHERE TODO;
'''
run_sql(sql)

In [ ]:
check_todo(
    16,
    sql,
    'USE school; SELECT ROUND(AVG(score),1) AS 90점이상평균 FROM students WHERE score >= 90;'
)

# 점수 미입력 학생 수 구하기

## TODO 17

`score IS NULL` 조건을 이용하여 점수가 입력되지 않은 학생 수를 구하세요.

In [ ]:
sql = '''
USE school;

SELECT TODO AS 점수미입력학생수
FROM students
WHERE TODO;
'''
run_sql(sql)

In [ ]:
check_todo(
    17,
    sql,
    'USE school; SELECT COUNT(*) AS 점수미입력학생수 FROM students WHERE score IS NULL;'
)

# 20. NULL과 집계 함수 정리

집계 함수에서 NULL은 보통 계산에서 제외됩니다.

| 함수 | NULL 처리 |
|---|---|
| `COUNT(*)` | NULL과 관계없이 행 전체를 셈 |
| `COUNT(score)` | NULL 제외 |
| `SUM(score)` | NULL 제외 |
| `AVG(score)` | NULL 제외 |
| `MAX(score)` | NULL 제외 |
| `MIN(score)` | NULL 제외 |

NULL을 0점으로 보고 싶다면:

```sql
IFNULL(score, 0)
```

을 사용할 수 있습니다.

> **NULL을 제외해서 계산할지, 0으로 바꾸어 계산할지는 의미가 서로 다릅니다.**

# 21. 자주 발생하는 실수

### 1) `COUNT(*)`와 `COUNT(score)`를 같은 것으로 생각하기
두 함수는 NULL이 있을 때 결과가 달라집니다.

### 2) `AVG(score)`가 NULL을 0으로 계산한다고 생각하기
`AVG(score)`는 NULL을 제외합니다.

### 3) 문자 컬럼에 `SUM()`이나 `AVG()`를 사용하기
`SUM`, `AVG`는 숫자 데이터에 사용하는 함수입니다.

### 4) 학과별 평균을 구하면서 GROUP BY를 빼먹기
학과별, 학년별처럼 **그룹별 집계**는 다음 시간에 `GROUP BY`로 배웁니다.

```sql
SELECT class_name, AVG(score)
FROM students
GROUP BY class_name;
```

# 22. 종합 실습

아래 문제는 짝 활동이나 마무리 문제로 활용하세요.

### 문제 A
전체 학생 수와 점수 입력 학생 수를 함께 출력하세요.

### 문제 B
전체 점수 합계와 평균 점수를 함께 출력하세요.

### 문제 C
최고 점수와 최저 점수를 함께 출력하세요.

### 문제 D
2학년 학생들의 평균 점수를 구하세요.

### 문제 E
80점 이상 학생의 수와 평균을 함께 출력하세요.

# 23. 오늘 실습 코드 전체 정리

```sql
SELECT COUNT(*) FROM students;

SELECT COUNT(score) FROM students;

SELECT SUM(score) FROM students;

SELECT AVG(score) FROM students;

SELECT ROUND(AVG(score), 1) AS 평균점수
FROM students;

SELECT MAX(score) FROM students;

SELECT MIN(score) FROM students;

SELECT
    COUNT(*) AS 전체학생수,
    COUNT(score) AS 점수입력수,
    SUM(score) AS 점수합계,
    ROUND(AVG(score), 1) AS 평균점수,
    MAX(score) AS 최고점수,
    MIN(score) AS 최저점수
FROM students;

SELECT COUNT(*)
FROM students
WHERE grade = 1;

SELECT ROUND(AVG(score), 1)
FROM students
WHERE grade = 1;

SELECT MAX(score)
FROM students
WHERE grade = 2;

SELECT MIN(score)
FROM students
WHERE grade = 3;

SELECT ROUND(AVG(IFNULL(score, 0)), 1)
FROM students;

SELECT COUNT(*)
FROM students
WHERE score >= 80;

SELECT ROUND(AVG(score), 1)
FROM students
WHERE score >= 90;

SELECT COUNT(*)
FROM students
WHERE score IS NULL;
```

# 24. 전체 TODO 진행 현황

In [ ]:
show_progress(17)

# 25. 오늘 배운 내용 정리

| 함수/문법 | 역할 |
|---|---|
| `COUNT(*)` | 전체 행 개수 |
| `COUNT(score)` | NULL이 아닌 score 개수 |
| `SUM(score)` | 점수 합계 |
| `AVG(score)` | 점수 평균 |
| `ROUND(..., 1)` | 소수점 첫째 자리까지 반올림 |
| `MAX(score)` | 가장 큰 값 |
| `MIN(score)` | 가장 작은 값 |
| `WHERE` | 조건에 맞는 행만 선택 |
| `IFNULL(score, 0)` | NULL을 0으로 바꾸어 계산 |

> **COUNT는 몇 개인지, SUM은 모두 더하기, AVG는 평균, MAX는 가장 큰 값, MIN은 가장 작은 값입니다.**

# 26. 자기 점검

- [ ] 집계 함수가 무엇인지 설명할 수 있다.
- [ ] `COUNT(*)`로 전체 학생 수를 구할 수 있다.
- [ ] `COUNT(*)`와 `COUNT(score)`의 차이를 설명할 수 있다.
- [ ] `SUM()`으로 점수 합계를 구할 수 있다.
- [ ] `AVG()`로 평균을 구할 수 있다.
- [ ] `ROUND()`로 평균을 보기 좋게 출력할 수 있다.
- [ ] `MAX()`와 `MIN()`을 사용할 수 있다.
- [ ] 여러 집계 함수를 한 SELECT문에서 사용할 수 있다.
- [ ] `WHERE`와 집계 함수를 함께 사용할 수 있다.
- [ ] NULL이 집계 함수에서 어떻게 처리되는지 설명할 수 있다.
- [ ] `IFNULL()`을 이용해 NULL을 0으로 보고 계산할 수 있다.

# 다음 시간 예고

## [SQL 실습 #10] MySQL GROUP BY 기초
### 학년별, 학과별로 묶어서 집계하기

다음 시간에는 오늘 배운 집계 함수를 전체에 한 번 적용하는 것을 넘어,

- 학년별 학생 수
- 학년별 평균 점수
- 학과별 학생 수
- 학과별 평균 점수

처럼 **기준별로 데이터를 묶어서 계산하는 방법**을 배웁니다.